# Analysis of the cosmology baseline $F$ problems

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

In [ ]:
feature = sympy.symbols("feature")
x1, x2 = sympy.symbols("x1:3")
x = sympy.abc.x
y = sympy.abc.y

## Load data

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
f_indices = full_report.data_set.apply(lambda x: x.startswith("F"))
fr1 = full_report[f_indices].sort_values("mse")
fr2 = fr1.set_index(["run_set", "data_set", "sample_num"])


In [ ]:
fr1.run_set.unique()

In [ ]:
fr2["sympy"] = fr2.expr_original_syms.apply(lambda e: au.parse_if_needed(e))

In [ ]:
fr2["complexity"] = fr2.sympy.apply(lambda e: au.complexity(e))


In [ ]:
fr2["sympy_defuzz"] = fr2.sympy.apply(lambda e: au.replace_near_integer(e.evalf()))


In [ ]:
fr2["complexity_defuzz"] = fr2.sympy_defuzz.apply(lambda e: au.complexity(e))

In [ ]:
fr2

In [ ]:
srb1 = fr2.loc["SRB-2026-06-26-1045-arr8"]
srb2 = fr2.loc["SRB-2026-07-13-1130"]
cht1 = fr2.loc["CHT-2026-07-02-1845"]
cht2 = fr2.loc["CHT-2026-07-13-1130"]
cht3 = fr2.loc["CHT-2026-06-28-2315"]

In [ ]:
(fr2.groupby(level=["run_set","data_set"]).size() == 32).all()

These are the best ones overall

In [ ]:
srb1_min_mse_ixs = srb1.groupby(level=["data_set"]).mse.idxmin()
srb2_min_mse_ixs = srb2.groupby(level=["data_set"]).mse.idxmin()
cht1_min_mse_ixs = cht1.groupby(level=["data_set"]).mse.idxmin()
cht2_min_mse_ixs = cht2.groupby(level=["data_set"]).mse.idxmin()
cht3_min_mse_ixs = cht3.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb1.loc[srb1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb2.loc[srb2_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht1.loc[cht1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht2.loc[cht2_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht3.loc[cht2_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb1.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

In [ ]:
srb2.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

In [ ]:
cht1.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

In [ ]:
cht2.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

It looks like `srb2` is slightly better at $F2$.
It looks like `cht1` is good but `cht2` is better.
I'm only keeping `cht3` because it solves $F_8$ once.

In [ ]:
srb = srb2
srb_min_mse_ixs = srb2_min_mse_ixs
cht = cht2
cht_min_mse_ixs = cht2_min_mse_ixs

## Polynomials

Generally, the polynomial problems are easy.

In [ ]:
data_sets_polynomial = ["F1", "F4"]

The best ones are exactly correct.

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_polynomial, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
au.count_by_threshold(srb.loc[data_sets_polynomial])

With some generous defuzzing, SRB gets all but one of the $F_1$'s exactly.

In [ ]:
(srb.loc["F1"]
    .sympy_defuzz
    .apply(lambda e:
           au.replace_near_integer(sympy.expand(e),
                                   tolerance=5e-3)))

In [ ]:
(cht.loc["F1"]
    .sympy_defuzz
    .apply(lambda e:
           au.replace_near_integer(sympy.expand(e),
                                   tolerance=5e-3)))

In [ ]:
srb.loc["F4"]

In [ ]:
srb.loc["F4",18]

Most of the $F_1$ solutions are polynomials.

In [ ]:
srb.loc["F1"].sympy_defuzz.apply(lambda e: e.is_polynomial(feature)).sum()

The $F_4$ solutions include a lot of cruft.

In [ ]:
srb.loc["F4"].sympy_defuzz.apply(lambda e: e.is_polynomial(x1, x2)).sum()

It looks like maybe 10 or so of these are perfectly correct with generous defuzzing.

In [ ]:
(srb.loc["F4"]
    .sympy_defuzz
    .apply(lambda e:
           au.replace_near_integer(sympy.expand(e),
                                   tolerance=5e-2)))

With CHT, looks like 20 or so are perfectly correct.

In [ ]:
(cht.loc["F4"]
    .sympy_defuzz
    .apply(lambda e:
           au.replace_near_integer(sympy.expand(e),
                                   tolerance=5e-3)))

In [ ]:
fig = sns.displot(data=srb.loc[data_sets_polynomial],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )
fig.set(ylim=(1e-30,1.0e5))

In [ ]:
fig = sns.displot(data=cht.loc[data_sets_polynomial],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True]
            )
fig.set(ylim=(1e-30,1.0e5))

## Rational functions

In [ ]:
data_sets_rational = ["F7"]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
au.count_by_threshold(srb.loc[data_sets_rational])

In [ ]:
au.count_by_threshold(cht.loc[data_sets_rational])

Only the first two of the SRB samples are arguably correct.

In [ ]:
srb.loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht.loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

About 10 or 12 of the CHT runs are arguable correct.

In [ ]:
cht.loc["F7"].sympy_defuzz.apply(
    lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1.0e-4))

In [ ]:
sns.displot(data=srb.loc[data_sets_rational],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

In [ ]:
sns.displot(data=cht.loc[data_sets_rational],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

## Medium: $F_5$

In [ ]:
data_sets_medium = ["F5"]

SRB gets all but two samples perfectly correct.
The initialization of the population is very important here.

In [ ]:
srb.loc["F5"]

My usual cheats don't help here, because I use a larger exponential and trig inventory, which leads to a lot of distraction and cruft that really damages the solutions.

In [ ]:
cht.loc["F5"]

In [ ]:
sns.displot(data=srb.loc[data_sets_medium],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

In [ ]:
sns.displot(data=cht.loc[data_sets_medium],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

## Hard: $F_2$

In [ ]:
srb1.loc["F2"]

In [ ]:
srb2.loc["F2"]

In [ ]:
cht1.loc["F2"]

In [ ]:
cht2.loc["F2"]

Only the `cht2` configuration comes close, and only once.

In [ ]:
sympy.expand(cht2.loc["F2", 23].sympy_defuzz.subs({feature: x}))

## Hard: $F_8$

In [ ]:
srb1.loc["F8"]

In [ ]:
srb2.loc["F8"]

In [ ]:
cht1.loc["F8"]

In [ ]:
cht2.loc["F8"]

In [ ]:
cht3.loc["F8", ["mse", "complexity_defuzz", "sympy_defuzz"]]

The `cht3` configuration solves $F_8$ ten times.

In [ ]:
au.count_by_threshold(cht3)

In [ ]:
sns.displot(data=cht3.loc["F8"],
            x="complexity",
            y="mse",
            log_scale=[False,True],
            )